In [3]:
import pandas as pd


In [4]:
# Load Reddit dataset
reddit = pd.read_csv("dog_whistle_reddit_sample_5000.csv")

# Load Silent Signals dictionary
glossary = pd.read_csv("silent_signals_reference_dictionary.csv")

In [5]:
print("Reddit dataset:")
print(reddit.shape)

print("\nSilent Signals dictionary:")
print(glossary.shape)

Reddit dataset:
(5000, 6)

Silent Signals dictionary:
(706, 4)


In [6]:
reddit.head()

,content,subreddit,dog_whistle,date,ingroup,source
0,"politically correct power grab (which , amusin...",MGTOW,politically correct,07-01-2019,racist,Pushift 2005-2022
1,Why do they take other people’s mere existence...,GenderCynical,GC,08/22/2018,transphobic,Pushift 2005-2022
2,&gt; Let's remember that this **includes our T...,GenderCynical,TIF,04-07-2020,transphobic,Pushift 2005-2022
3,The gay Mafia. It’s all about societal degrada...,MGTOW,public schools,04-05-2020,racist,Pushift 2005-2022
4,Single mother - A mom who prioritises her own ...,MGTOW,single mother,03/17/2020,racist,Pushift 2005-2022


In [7]:
reddit.columns

Index(['content', 'subreddit', 'dog_whistle', 'date', 'ingroup', 'source'], dtype='object')

In [8]:
reddit.isnull().sum()

content        0
subreddit      0
dog_whistle    0
date           0
ingroup        0
source         0
dtype: int64

In [9]:
reddit["content"]


0       politically correct power grab (which , amusin...
1       Why do they take other people’s mere existence...
2       &gt; Let's remember that this **includes our T...
3       The gay Mafia. It’s all about societal degrada...
4       Single mother - A mom who prioritises her own ...
                              ...                        
4995    Women think that by acting like shitty men who...
4996    It's such a dissonance to hear them talk about...
4997    Omg- individual stalls with a shower and chang...
4998    I recall a TERF fearmongering about a dystopia...
4999    So he's not allowed to have feelings about thi...
Name: content, Length: 5000, dtype: object

In [10]:
# Display the first 5 Reddit posts
reddit["content"].head()

0    politically correct power grab (which , amusin...
1    Why do they take other people’s mere existence...
2    &gt; Let's remember that this **includes our T...
3    The gay Mafia. It’s all about societal degrada...
4    Single mother - A mom who prioritises her own ...
Name: content, dtype: object

In [11]:
reddit.isnull().sum()


content        0
subreddit      0
dog_whistle    0
date           0
ingroup        0
source         0
dtype: int64

In [12]:
# Count empty or whitespace-only posts
empty_posts = reddit["content"].fillna("").str.strip().eq("").sum()

print("Empty posts:", empty_posts)

Empty posts: 0


In [13]:
empty_posts = reddit["content"].str.strip().eq("").sum()

print("Empty posts:", empty_posts)

Empty posts: 0


In [14]:
for i, post in enumerate(reddit["content"].head(5), 1):
    print(f"\nPOST {i}:")
    print(post)


POST 1:
politically correct power grab (which , amusingly, was popularized by multi million dollar net worth Hollywood celebrities who live in mansions) now backfiring.

POST 2:
Why do they take other people’s mere existence as a personal affront? If you’re not interested, just swipe left and move on 💁🏻‍♀️ &gt;Well we are being erased so this makes perfect sense when you look at it from that perspective. How can we continue to date if our dating pool is flooded with trash? GC or incel? 🤔

POST 3:
&gt; Let's remember that this **includes our TIF sisters** who - while deeply misguided to the cause - are not allowed to express their masculinity or sexuality without risk.

POST 4:
The gay Mafia. It’s all about societal degradation. They want to turn everyone gay. Or at if your not gay you should at least wear a dress. And if your gonna wear a dress you should at least go hang around confused little kids in public schools.

POST 5:
Single mother - A mom who prioritises her own comfort more

In [15]:
import re
import html


In [16]:
def clean_text(text):

    # Convert HTML entities into normal characters
    text = html.unescape(str(text))

    # Convert everything to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)

    # Normalize curly apostrophes
    text = text.replace("’", "'")

    # Remove markdown formatting symbols
    text = re.sub(r"[*_~`]", "", text)

    # Keep words and apostrophes, replace other characters with spaces
    text = re.sub(r"[^a-zA-Z']+", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text
    

In [17]:
original_post = reddit["content"].iloc[0]

cleaned_post = clean_text(original_post)

print("ORIGINAL POST:")
print(original_post)

print("\nCLEANED POST:")
print(cleaned_post)

ORIGINAL POST:
politically correct power grab (which , amusingly, was popularized by multi million dollar net worth Hollywood celebrities who live in mansions) now backfiring.

CLEANED POST:
politically correct power grab which amusingly was popularized by multi million dollar net worth hollywood celebrities who live in mansions now backfiring


In [18]:
reddit["cleaned_content"] = reddit["content"].apply(clean_text)

In [19]:
reddit["tokens"] = reddit["cleaned_content"].apply(
    lambda text: text.split()
)

In [20]:
reddit[["content", "cleaned_content", "tokens"]].head()


,content,cleaned_content,tokens
0,"politically correct power grab (which , amusin...",politically correct power grab which amusingly...,"[politically, correct, power, grab, which, amu..."
1,Why do they take other people’s mere existence...,why do they take other people's mere existence...,"[why, do, they, take, other, people's, mere, e..."
2,&gt; Let's remember that this **includes our T...,let's remember that this includes our tif sist...,"[let's, remember, that, this, includes, our, t..."
3,The gay Mafia. It’s all about societal degrada...,the gay mafia it's all about societal degradat...,"[the, gay, mafia, it's, all, about, societal, ..."
4,Single mother - A mom who prioritises her own ...,single mother a mom who prioritises her own co...,"[single, mother, a, mom, who, prioritises, her..."


In [21]:
def extract_ngrams(tokens, n):

    ngrams = []

    for i in range(len(tokens) - n + 1):

        expression = " ".join(tokens[i:i+n])

        ngrams.append(expression)

    return ngrams

In [22]:
sample_tokens = [
    "protect",
    "our",
    "western",
    "civilization"
]

print("Unigrams:")
print(extract_ngrams(sample_tokens, 1))

print("\nBigrams:")
print(extract_ngrams(sample_tokens, 2))

print("\nTrigrams:")
print(extract_ngrams(sample_tokens, 3))

Unigrams:
['protect', 'our', 'western', 'civilization']

Bigrams:
['protect our', 'our western', 'western civilization']

Trigrams:
['protect our western', 'our western civilization']


In [23]:
from collections import Counter, defaultdict

# Create counters for each n-gram type
unigram_counts = Counter()
bigram_counts = Counter()
trigram_counts = Counter()

# Store example posts
ngram_examples = defaultdict(list)

# Process every Reddit post
for index, tokens in enumerate(reddit["tokens"]):

    # Extract unigrams
    unigrams = extract_ngrams(tokens, 1)
    unigram_counts.update(unigrams)

    # Extract bigrams
    bigrams = extract_ngrams(tokens, 2)
    bigram_counts.update(bigrams)

    # Extract trigrams
    trigrams = extract_ngrams(tokens, 3)
    trigram_counts.update(trigrams)

    # Store examples for each expression
    for n, expressions in [
        (1, unigrams),
        (2, bigrams),
        (3, trigrams)
    ]:

        for expression in set(expressions):

            if len(ngram_examples[(n, expression)]) < 3:

                ngram_examples[(n, expression)].append(
                    reddit["content"].iloc[index]
                )

In [24]:
print("Unique unigrams:", len(unigram_counts))
print("Unique bigrams:", len(bigram_counts))
print("Unique trigrams:", len(trigram_counts))

Unique unigrams: 15068
Unique bigrams: 108792
Unique trigrams: 183758


In [25]:
print("TOP 20 UNIGRAMS:")
print(unigram_counts.most_common(20))

print("\nTOP 20 BIGRAMS:")
print(bigram_counts.most_common(20))

print("\nTOP 20 TRIGRAMS:")
print(trigram_counts.most_common(20))

TOP 20 UNIGRAMS:
[('the', 7744), ('to', 5853), ('and', 5545), ('a', 5522), ('of', 4034), ('i', 3521), ('is', 3282), ('that', 3085), ('in', 2594), ('you', 2313), ('they', 2103), ('it', 2051), ('are', 2049), ('for', 1964), ('on', 1663), ('with', 1562), ('be', 1425), ('this', 1415), ('have', 1404), ('but', 1333)]

TOP 20 BIGRAMS:
[('of the', 576), ('in the', 573), ('to be', 517), ('is a', 390), ('a single', 331), ('based on', 328), ('on the', 296), ('this is', 280), ('as a', 276), ('they are', 268), ('to the', 263), ('if you', 249), ('the same', 231), ("i don't", 225), ('for the', 223), ('want to', 216), ('and the', 213), ('is the', 205), ('in a', 199), ('a lot', 195)]

TOP 20 TRIGRAMS:
[('a lot of', 135), ('to be a', 67), ('one of the', 53), ('a single mom', 52), ('a single mother', 51), ('this is a', 42), ("i don't know", 42), ('based on the', 42), ('the fact that', 37), ('is based on', 33), ('the rest of', 33), ('the blue pill', 33), ('to do with', 32), ('there is no', 31), ('there is 

In [26]:
MIN_FREQUENCY = 5

# Filter unigrams
filtered_unigrams = Counter({
    expression: count
    for expression, count in unigram_counts.items()
    if count >= MIN_FREQUENCY
})

# Filter bigrams
filtered_bigrams = Counter({
    expression: count
    for expression, count in bigram_counts.items()
    if count >= MIN_FREQUENCY
})

# Filter trigrams
filtered_trigrams = Counter({
    expression: count
    for expression, count in trigram_counts.items()
    if count >= MIN_FREQUENCY
})

print("Unigrams remaining:", len(filtered_unigrams))
print("Bigrams remaining:", len(filtered_bigrams))
print("Trigrams remaining:", len(filtered_trigrams))

Unigrams remaining: 3638
Bigrams remaining: 5801
Trigrams remaining: 1364


In [27]:
print("FREQUENCY FILTERING SUMMARY")

print("Unigrams:")
print("Before:", len(unigram_counts))
print("After:", len(filtered_unigrams))

print("\nBigrams:")
print("Before:", len(bigram_counts))
print("After:", len(filtered_bigrams))

print("\nTrigrams:")
print("Before:", len(trigram_counts))
print("After:", len(filtered_trigrams))

FREQUENCY FILTERING SUMMARY
Unigrams:
Before: 15068
After: 3638

Bigrams:
Before: 108792
After: 5801

Trigrams:
Before: 183758
After: 1364


In [28]:
glossary.head()

,dog_whistle,dog_whistle_root,definition,ingroup
0,SJW,social justice warrior,liberals,anti-liberal
1,SJWs,social justice warrior,liberals,anti-liberal
2,blue,blueish,Jewish,antisemitic
3,LGB,LGB rights,"Lesbian, gay, and bisexual, but not trans, rights",transphobic
4,working class,working class,white working class,racist


In [29]:
known_expressions = set()

for column in ["dog_whistle", "dog_whistle_root"]:

    for term in glossary[column].dropna():

        normalized = " ".join(clean_text(term).split())

        if normalized:
            known_expressions.add(normalized)

print("Unique glossary expressions:", len(known_expressions))

Unique glossary expressions: 691


In [30]:
all_filtered = {
    1: filtered_unigrams,
    2: filtered_bigrams,
    3: filtered_trigrams
}

results = []

for n, counter in all_filtered.items():

    for expression, count in counter.most_common():

        status = (
            "Already in glossary"
            if expression in known_expressions
            else "Not in glossary"
        )

        results.append({

            "expression": expression,
            "ngram_type": n,
            "frequency": count,
            "glossary_status": status,

            "example_post": (
                ngram_examples[(n, expression)][0]
                if ngram_examples[(n, expression)]
                else ""
            )
        })

results_df = pd.DataFrame(results)

# Rank by frequency
results_df = results_df.sort_values(
    "frequency",
    ascending=False
).reset_index(drop=True)

results_df.head(20)

,expression,ngram_type,frequency,glossary_status,example_post
0,the,1,7744,Not in glossary,&gt; Let's remember that this **includes our T...
1,to,1,5853,Not in glossary,Why do they take other people’s mere existence...
2,and,1,5545,Not in glossary,Why do they take other people’s mere existence...
3,a,1,5522,Not in glossary,Why do they take other people’s mere existence...
4,of,1,4034,Not in glossary,Single mother - A mom who prioritises her own ...
5,i,1,3521,Not in glossary,Times are different. Women dump men when there...
6,is,1,3282,Not in glossary,Why do they take other people’s mere existence...
7,that,1,3085,Not in glossary,Why do they take other people’s mere existence...
8,in,1,2594,Not in glossary,"politically correct power grab (which , amusin..."
9,you,1,2313,Not in glossary,Why do they take other people’s mere existence...


In [31]:
print(results_df["glossary_status"].value_counts())

glossary_status
Not in glossary        10678
Already in glossary      125
Name: count, dtype: int64


In [32]:
new_candidates = results_df[
    results_df["glossary_status"] == "Not in glossary"
]

new_candidates.head(30)

,expression,ngram_type,frequency,glossary_status,example_post
0,the,1,7744,Not in glossary,&gt; Let's remember that this **includes our T...
1,to,1,5853,Not in glossary,Why do they take other people’s mere existence...
2,and,1,5545,Not in glossary,Why do they take other people’s mere existence...
3,a,1,5522,Not in glossary,Why do they take other people’s mere existence...
4,of,1,4034,Not in glossary,Single mother - A mom who prioritises her own ...
5,i,1,3521,Not in glossary,Times are different. Women dump men when there...
6,is,1,3282,Not in glossary,Why do they take other people’s mere existence...
7,that,1,3085,Not in glossary,Why do they take other people’s mere existence...
8,in,1,2594,Not in glossary,"politically correct power grab (which , amusin..."
9,you,1,2313,Not in glossary,Why do they take other people’s mere existence...


In [33]:
# Show the 30 most frequent bigrams
print("TOP 30 BIGRAMS")
display(
    results_df[results_df["ngram_type"] == 2]
    [["expression", "frequency", "glossary_status"]]
    .head(30)
)

# Show the 30 most frequent trigrams
print("TOP 30 TRIGRAMS")
display(
    results_df[results_df["ngram_type"] == 3]
    [["expression", "frequency", "glossary_status"]]
    .head(30)
)


TOP 30 BIGRAMS


,expression,frequency,glossary_status
61,of the,576,Not in glossary
64,in the,573,Not in glossary
71,to be,517,Not in glossary
82,is a,390,Not in glossary
97,a single,331,Not in glossary
98,based on,328,Not in glossary
108,on the,296,Not in glossary
120,this is,280,Not in glossary
122,as a,276,Not in glossary
125,they are,268,Not in glossary


TOP 30 TRIGRAMS


,expression,frequency,glossary_status
262,a lot of,135,Not in glossary
549,to be a,67,Not in glossary
687,one of the,53,Not in glossary
713,a single mom,52,Not in glossary
715,a single mother,51,Not in glossary
883,based on the,42,Not in glossary
889,i don't know,42,Not in glossary
906,this is a,42,Not in glossary
1042,the fact that,37,Not in glossary
1148,is based on,33,Not in glossary


In [34]:
# Select expressions we want to investigate
expressions_to_inspect = [
    (2, "trans people"),
    (2, "a woman"),
    (3, "a single mother"),
    (3, "the blue pill")
]

for n, expression in expressions_to_inspect:
    print("\n" + "=" * 70)
    print(f"Expression: {expression} | N-gram type: {n}")
    print("=" * 70)

    examples = ngram_examples.get((n, expression), [])

    if examples:
        for i, post in enumerate(examples[:3], start=1):
            print(f"\nExample {i}:\n{post}")
    else:
        print("No examples found.")


Expression: trans people | N-gram type: 2

Example 1:
There seems to be a handful of GCers who use their alts there to dole out abuse to the few trans people who stick around.

Example 2:
“I’m being silenced” = “people are disagreeing with my opinion and calling me things I don’t like!” Like, I’m willing to listen to this rhetoric, but I need a reason for how women will be harmed by others being allowed to legally identify as their preferred gender. Cause so far not only do I not see how women are or will be harmed.... I am seeing trans people being harmed right now as a result of this rhetoric.

Example 3:
I've yet to see a "separate place" for LGB people that actually focus on LGB problems. They just complain about trans people all day. A bit like MGTOW, actually.

Expression: a woman | N-gram type: 2

Example 1:
quite the opposite. they are barking now because they fear that this would happen to them or what not. they are literally one bj away from plantation and at the very moment

In [35]:
# Find Reddit posts containing "the blue pill"

blue_pill_posts = reddit[
    reddit["cleaned_content"].str.contains(
        r"\bthe blue pill\b",
        regex=True,
        na=False
    )
]

# Count occurrences by subreddit
blue_pill_subreddits = (
    blue_pill_posts["subreddit"]
    .value_counts()
    .reset_index()
)

blue_pill_subreddits.columns = [
    "subreddit",
    "number_of_posts"
]

display(blue_pill_subreddits.head(15))

print("Total matching posts:", len(blue_pill_posts))

,subreddit,number_of_posts
0,MGTOW,32


Total matching posts: 32


In [36]:
# Display the relevant information for all matching posts

display(
    blue_pill_posts[
        ["subreddit", "ingroup", "dog_whistle", "content"]
    ].reset_index(drop=True)
)

,subreddit,ingroup,dog_whistle,content
0,MGTOW,antisemitic,blue,It's a good intro if you're fresh off the blue...
1,MGTOW,antisemitic,blue,I feel for you guys in the US. Over here in Sw...
2,MGTOW,antisemitic,blue,Yo if a feminist was to see this picture they ...
3,MGTOW,antisemitic,blue,"I agree with what you say, when i first decide..."
4,MGTOW,antisemitic,blue,For the love of god keep your kids the fuck aw...
5,MGTOW,antisemitic,blue,They are not sorry. These thots are just virtu...
6,MGTOW,antisemitic,blue,You got off clean before she got her claws int...
7,MGTOW,antisemitic,blue,Hes also horribly depressed and emotionally cr...
8,MGTOW,antisemitic,blue,There is still the blue pill mentality that yo...
9,MGTOW,antisemitic,blue,Said the religion basing their worship on a Je...


In [37]:
# Check the annotation distribution across all 32 posts

print("Ingroup labels:")
display(blue_pill_posts["ingroup"].value_counts(dropna=False))

print("\nDog-whistle labels:")
display(blue_pill_posts["dog_whistle"].value_counts(dropna=False))

print("\nSubreddit distribution:")
display(blue_pill_posts["subreddit"].value_counts(dropna=False))

Ingroup labels:


ingroup
antisemitic          31
white supremacist     1
Name: count, dtype: int64


Dog-whistle labels:


dog_whistle
blue     31
cucks     1
Name: count, dtype: int64


Subreddit distribution:


subreddit
MGTOW    32
Name: count, dtype: int64

In [38]:
for index, post in enumerate(blue_pill_posts["content"], start=1):
    print(f"\nPOST {index}")
    print(post[:700])


POST 1
It's a good intro if you're fresh off the blue pill. Otherwise it's a rehash of the standard RP fare.

POST 2
I feel for you guys in the US. Over here in Sweden the only required amount to pay is 140$ a month per kid. And if you split custody 50/50 you don't have to pay a thing. Looks like someone set up your system to kill the financial situation of the blue pill-men who think with their penis.

POST 3
Yo if a feminist was to see this picture they would start ww3, look at those poor weak totally not independent Women who need a man to Simp for them. The Blue Pill does get us young.

POST 4
I agree with what you say, when i first decided to become a farmer I was the Mayor of the blue pill zone,Only once i actually got into it ,I realised how hard it is.

POST 5
For the love of god keep your kids the fuck away from anything Disney, those fuckers are the epitome of the blue pill, brainwashing at its finest. Please don’t support those mofos

POST 6
They are not sorry. These thots 

In [39]:
print("Ingroup labels:")
display(blue_pill_posts["ingroup"].value_counts(dropna=False))

print("\nDog-whistle labels:")
display(blue_pill_posts["dog_whistle"].value_counts(dropna=False))

print("\nSubreddit labels:")
display(blue_pill_posts["subreddit"].value_counts(dropna=False))

Ingroup labels:


ingroup
antisemitic          31
white supremacist     1
Name: count, dtype: int64


Dog-whistle labels:


dog_whistle
blue     31
cucks     1
Name: count, dtype: int64


Subreddit labels:


subreddit
MGTOW    32
Name: count, dtype: int64

In [40]:
# Find the one post with a different dog-whistle annotation

exception_post = blue_pill_posts[
    blue_pill_posts["dog_whistle"] == "cucks"
]

display(
    exception_post[
        ["subreddit", "ingroup", "dog_whistle", "content"]
    ]
)

,subreddit,ingroup,dog_whistle,content
2426,MGTOW,white supremacist,cucks,"Look at the blue-pill cucks of today's world, ..."


In [41]:
# Search the glossary for these two terms

glossary[
    glossary["dog_whistle"].str.contains(
        r"\b(blue|cucks)\b",
        case=False,
        regex=True,
        na=False
    )
][["dog_whistle", "dog_whistle_root", "definition", "ingroup"]]

/var/folders/1c/hp8bbnfn0h53mmlvb7jgzqtr0000gn/T/ipykernel_4085/3967375732.py:4: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  glossary["dog_whistle"].str.contains(


,dog_whistle,dog_whistle_root,definition,ingroup
2,blue,blueish,Jewish,antisemitic
7,cucks,cuck,liberal or establishment Republican men,white supremacist
214,Blue Lives Matter,Blue Lives Matter,Black lives don't matter,racist
342,thin blue line flag,thin blue line,Protect white people from non-white people,racist
596,Back the Blue,Back the Blue,Enforce white supremacy,racist
682,thin blue line,thin blue line,Protect white people from non-white people,racist


In [42]:
glossary[
    glossary["dog_whistle"].str.contains(
        r"\b(blue|cucks)\b",
        case=False,
        regex=True,
        na=False
    )
][["dog_whistle", "dog_whistle_root", "definition", "ingroup"]]

/var/folders/1c/hp8bbnfn0h53mmlvb7jgzqtr0000gn/T/ipykernel_4085/2707068139.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  glossary["dog_whistle"].str.contains(


,dog_whistle,dog_whistle_root,definition,ingroup
2,blue,blueish,Jewish,antisemitic
7,cucks,cuck,liberal or establishment Republican men,white supremacist
214,Blue Lives Matter,Blue Lives Matter,Black lives don't matter,racist
342,thin blue line flag,thin blue line,Protect white people from non-white people,racist
596,Back the Blue,Back the Blue,Enforce white supremacy,racist
682,thin blue line,thin blue line,Protect white people from non-white people,racist


In [43]:
glossary[
    glossary["dog_whistle"].str.contains(
        r"\b(?:blue|cucks)\b",
        case=False,
        regex=True,
        na=False
    )
][["dog_whistle", "dog_whistle_root", "definition", "ingroup"]]

,dog_whistle,dog_whistle_root,definition,ingroup
2,blue,blueish,Jewish,antisemitic
7,cucks,cuck,liberal or establishment Republican men,white supremacist
214,Blue Lives Matter,Blue Lives Matter,Black lives don't matter,racist
342,thin blue line flag,thin blue line,Protect white people from non-white people,racist
596,Back the Blue,Back the Blue,Enforce white supremacy,racist
682,thin blue line,thin blue line,Protect white people from non-white people,racist


In [44]:
from collections import defaultdict

document_frequency = defaultdict(int)

for tokens in reddit["tokens"]:
    for n in [1, 2, 3]:
        expressions = set(extract_ngrams(tokens, n))

        for expression in expressions:
            document_frequency[(n, expression)] += 1

In [45]:
results_df["document_frequency"] = results_df.apply(
    lambda row: document_frequency[
        (row["ngram_type"], row["expression"])
    ],
    axis=1
)

display(
    results_df.sort_values(
        "frequency",
        ascending=False
    ).head(20)
)

,expression,ngram_type,frequency,glossary_status,example_post,document_frequency
0,the,1,7744,Not in glossary,&gt; Let's remember that this **includes our T...,3593
1,to,1,5853,Not in glossary,Why do they take other people’s mere existence...,3247
2,and,1,5545,Not in glossary,Why do they take other people’s mere existence...,3237
3,a,1,5522,Not in glossary,Why do they take other people’s mere existence...,3127
4,of,1,4034,Not in glossary,Single mother - A mom who prioritises her own ...,2644
5,i,1,3521,Not in glossary,Times are different. Women dump men when there...,2003
6,is,1,3282,Not in glossary,Why do they take other people’s mere existence...,2239
7,that,1,3085,Not in glossary,Why do they take other people’s mere existence...,2151
8,in,1,2594,Not in glossary,"politically correct power grab (which , amusin...",1915
9,you,1,2313,Not in glossary,Why do they take other people’s mere existence...,1371


In [46]:
# Show the most widely distributed bigrams
print("TOP 20 BIGRAMS BY DOCUMENT FREQUENCY")

display(
    results_df[results_df["ngram_type"] == 2]
    .sort_values("document_frequency", ascending=False)
    [["expression", "frequency",
      "document_frequency", "glossary_status"]]
    .head(20)
)

# Show the most widely distributed trigrams
print("TOP 20 TRIGRAMS BY DOCUMENT FREQUENCY")

display(
    results_df[results_df["ngram_type"] == 3]
    .sort_values("document_frequency", ascending=False)
    [["expression", "frequency",
      "document_frequency", "glossary_status"]]
    .head(20)
)

TOP 20 BIGRAMS BY DOCUMENT FREQUENCY


,expression,frequency,document_frequency,glossary_status
64,in the,573,522,Not in glossary
61,of the,576,517,Not in glossary
71,to be,517,473,Not in glossary
82,is a,390,363,Not in glossary
98,based on,328,318,Not in glossary
97,a single,331,317,Not in glossary
108,on the,296,278,Not in glossary
120,this is,280,258,Not in glossary
122,as a,276,254,Not in glossary
127,to the,263,252,Not in glossary


TOP 20 TRIGRAMS BY DOCUMENT FREQUENCY


,expression,frequency,document_frequency,glossary_status
262,a lot of,135,130,Not in glossary
549,to be a,67,64,Not in glossary
687,one of the,53,52,Not in glossary
715,a single mother,51,51,Not in glossary
713,a single mom,52,50,Not in glossary
883,based on the,42,42,Not in glossary
906,this is a,42,42,Not in glossary
889,i don't know,42,41,Not in glossary
1042,the fact that,37,36,Not in glossary
1148,is based on,33,33,Not in glossary


In [47]:
# Select bigrams and trigrams
phrase_candidates = results_df[
    results_df["ngram_type"].isin([2, 3])
].copy()

# Add the percentage of posts containing each expression
phrase_candidates["document_percentage"] = (
    phrase_candidates["document_frequency"] / len(reddit) * 100
).round(2)

# Sort by document frequency
phrase_candidates = phrase_candidates.sort_values(
    ["document_frequency", "frequency"],
    ascending=False
).reset_index(drop=True)

# Display the top 50 candidates
display(
    phrase_candidates[
        [
            "expression",
            "ngram_type",
            "frequency",
            "document_frequency",
            "document_percentage",
            "glossary_status",
            "example_post"
        ]
    ].head(50)
)

,expression,ngram_type,frequency,document_frequency,document_percentage,glossary_status,example_post
0,in the,2,573,522,10.44,Not in glossary,"As an Indian, it is really unfortunate to see ..."
1,of the,2,576,517,10.34,Not in glossary,And dealt with stuff like having to pay 100% o...
2,to be,2,517,473,9.46,Not in glossary,There seems to be a handful of GCers who use t...
3,is a,2,390,363,7.26,Not in glossary,Single mother - A mom who prioritises her own ...
4,based on,2,328,318,6.36,Not in glossary,Diversity hiring is such bullshit. Companies s...
5,a single,2,331,317,6.34,Not in glossary,Single mother - A mom who prioritises her own ...
6,on the,2,296,278,5.56,Not in glossary,Do you know what post the GC advocated for sho...
7,this is,2,280,258,5.16,Not in glossary,"I'm be honest, I kinda had the same concerns s..."
8,as a,2,276,254,5.08,Not in glossary,Why do they take other people’s mere existence...
9,to the,2,263,252,5.04,Not in glossary,&gt; Let's remember that this **includes our T...


In [48]:
# Save the complete ranked results

results_df.to_csv(
    "frequency_analysis_results.csv",
    index=False
)

print("Results successfully saved!")

Results successfully saved!
